<a href="https://colab.research.google.com/github/hoapp07/FaceID_AI_UEH/blob/main/FaceID2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PHẦN 1: HUẤN LUYỆN MÔ HÌNH - Dùng Transfer Learning (MobileNetV2)
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import os

# Thiết lập đường dẫn
train_dir = "/content/drive/MyDrive/60 tấm selfies - sáng T2"
model_save_path = "/content/drive/MyDrive/face_recognition_model.h5"
class_indices_path = "/content/drive/MyDrive/class_indices.npy"

img_width, img_height = 200, 200
batch_size = 16
NUM_CLASSES = 22

# TIỀN XỬ LÝ + TĂNG CƯỜNG DỮ LIỆU
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='training',
    shuffle=True,
    seed=42
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset='validation',
    shuffle=False,
    seed=42
)

# Lưu class indices để dùng lại sau
np.save(class_indices_path, train_generator.class_indices)
print("Các lớp đã phát hiện:", train_generator.class_indices)
print(f"Tổng ảnh training: {train_generator.samples}")
print(f"Tổng ảnh validation: {validation_generator.samples}")


# XÂY DỰNG MÔ HÌNH VỚI TRANSFER LEARNING (MobileNetV2)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(img_width, img_height, 3)
)

# Đóng băng base model ban đầu
base_model.trainable = False

# Xây dựng model
inputs = Input(shape=(img_width, img_height, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# CALLBACKS
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

# GIAI ĐOẠN 1: Huấn luyện phần đầu (frozen base)
print("\n=== GIAI ĐOẠN 1: Huấn luyện classifier ===")
steps_per_epoch = max(1, train_generator.samples // batch_size)
validation_steps = max(1, validation_generator.samples // batch_size)

history1 = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks
)

# GIAI ĐOẠN 2: Fine-tuning
print("\n=== GIAI ĐOẠN 2: Fine-tuning ===")
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks
)
# VẼ BIỂU ĐỒ
acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='Train Accuracy')
plt.plot(val_acc, label='Val Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.title('Độ chính xác'); plt.legend()

plt.subplot(1, 2, 2)
loss = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
plt.plot(loss, label='Train Loss')
plt.plot(val_loss, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()

print(f"\n✅ Model đã lưu tại: {model_save_path}")

In [ ]:
# PHẦN 2: KIỂM TRA ĐƠN LẺ (thay thế cell cũ)
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img
import numpy as np
import matplotlib.pyplot as plt

model_save_path = "/content/drive/MyDrive/face_recognition_model.h5"
class_indices_path = "/content/drive/MyDrive/class_indices.npy"
img_width, img_height = 200, 200
CONFIDENCE_THRESHOLD = 0.5 # Ngưỡng tin cậy

# Load model và class indices
model = load_model(model_save_path)
class_indices = np.load(class_indices_path, allow_pickle=True).item()
class_labels = {v: k for k, v in class_indices.items()}

def predict_face(path):
    img = load_img(path, target_size=(img_width, img_height))
    plt.imshow(img); plt.axis('off'); plt.show()

    img_array = np.array(img) / 255.0
    img_array = img_array.reshape(1, img_width, img_height, 3)

    predictions = model.predict(img_array)[0]
    top_idx = np.argmax(predictions)
    confidence = predictions[top_idx]
    person_name = class_labels[top_idx]
    print(f"Người tiên đoán: {person_name}")


# Test
path = '/content/drive/MyDrive/60 tấm selfies - sáng T2/Phạm Phú Hoà/IMG_7030.PNG'
predict_face(path)

In [ ]:
# PHẦN 3: TẠO WEB APP - Chạy cell này trong Colab
!pip install flask flask-cors pyngrok pillow tensorflow -q

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import tensorflow as tf
import numpy as np
from PIL import Image
import base64
import io
import os

app = Flask(__name__)
CORS(app)

# Load model
MODEL_PATH = "/content/drive/MyDrive/face_recognition_model.h5"
CLASS_INDICES_PATH = "/content/drive/MyDrive/class_indices.npy"

model = tf.keras.models.load_model(MODEL_PATH)
class_indices = np.load(CLASS_INDICES_PATH, allow_pickle=True).item()
class_labels = {v: k for k, v in class_indices.items()}

IMG_SIZE = 200
CONFIDENCE_THRESHOLD = 0.5

def preprocess_image(image_data):
    """Xử lý ảnh từ base64"""
    if ',' in image_data:
        image_data = image_data.split(',')[1]
    img_bytes = base64.b64decode(image_data)
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img) / 255.0
    return img_array.reshape(1, IMG_SIZE, IMG_SIZE, 3)

@app.route('/')
def index():
    return '''<!DOCTYPE html>
<html lang="vi">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Face ID - Điểm Danh Lớp</title>
<style>
  * { margin: 0; padding: 0; box-sizing: border-box; }
  body { font-family: 'Segoe UI', sans-serif; background: #0a0a1a; color: #fff; min-height: 100vh; }
  .header { background: linear-gradient(135deg, #1a1a3e, #2d2d6b); padding: 20px; text-align: center; border-bottom: 2px solid #4a4aff; }
  .header h1 { font-size: 28px; color: #7b7bff; }
  .header p { color: #aaa; margin-top: 5px; }
  .container { max-width: 900px; margin: 30px auto; padding: 0 20px; }
  .camera-section { background: #12122a; border-radius: 16px; padding: 25px; margin-bottom: 25px; border: 1px solid #2a2a5a; }
  .camera-wrapper { position: relative; width: 100%; max-width: 500px; margin: 0 auto; }
  video, canvas { width: 100%; border-radius: 12px; border: 3px solid #2a2a5a; display: block; }
  canvas { display: none; }
  .scan-line { position: absolute; top: 0; left: 0; right: 0; height: 3px; background: linear-gradient(90deg, transparent, #4a4aff, transparent); animation: scan 2s linear infinite; pointer-events: none; display: none; }
  @keyframes scan { 0% { top: 0; } 100% { top: 100%; } }
  .scanning .scan-line { display: block; }
  .btn-group { display: flex; gap: 12px; justify-content: center; margin-top: 20px; flex-wrap: wrap; }
  .btn { padding: 12px 28px; border: none; border-radius: 10px; font-size: 15px; font-weight: 600; cursor: pointer; transition: all 0.3s; }
  .btn-primary { background: linear-gradient(135deg, #4a4aff, #7b7bff); color: white; }
  .btn-primary:hover { transform: translateY(-2px); box-shadow: 0 8px 25px rgba(74,74,255,0.4); }
  .btn-success { background: linear-gradient(135deg, #00b894, #00cec9); color: white; }
  .btn-success:hover { transform: translateY(-2px); box-shadow: 0 8px 25px rgba(0,184,148,0.4); }
  .btn-danger { background: linear-gradient(135deg, #e17055, #d63031); color: white; }
  .btn:disabled { opacity: 0.5; cursor: not-allowed; transform: none !important; }
  .result-section { background: #12122a; border-radius: 16px; padding: 25px; border: 1px solid #2a2a5a; }
  .result-card { background: #1a1a3e; border-radius: 12px; padding: 20px; text-align: center; margin-top: 15px; display: none; }
  .result-card.show { display: block; animation: fadeIn 0.5s ease; }
  @keyframes fadeIn { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
  .result-name { font-size: 26px; font-weight: bold; margin: 10px 0; }
  .result-confidence { font-size: 16px; color: #aaa; margin-bottom: 10px; }
  .confidence-bar { background: #2a2a5a; border-radius: 10px; height: 10px; overflow: hidden; margin: 8px 0; }
  .confidence-fill { height: 100%; border-radius: 10px; transition: width 1s ease; }
  .status-known { color: #00b894; border: 2px solid #00b894; }
  .status-unknown { color: #e17055; border: 2px solid #e17055; }
  .top3 { margin-top: 15px; }
  .top3-item { display: flex; justify-content: space-between; align-items: center; padding: 8px 12px; background: #12122a; border-radius: 8px; margin-bottom: 6px; font-size: 14px; }
  .attendance-log { background: #12122a; border-radius: 16px; padding: 25px; border: 1px solid #2a2a5a; margin-top: 25px; }
  .log-table { width: 100%; border-collapse: collapse; margin-top: 15px; }
  .log-table th { background: #1a1a3e; padding: 10px; text-align: left; color: #7b7bff; font-size: 13px; }
  .log-table td { padding: 10px; border-bottom: 1px solid #2a2a5a; font-size: 14px; }
  .log-table tr:hover td { background: #1a1a3e; }
  .badge { padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight: bold; }
  .badge-present { background: rgba(0,184,148,0.2); color: #00b894; }
  .badge-unknown { background: rgba(225,112,85,0.2); color: #e17055; }
  .section-title { font-size: 18px; font-weight: bold; color: #7b7bff; margin-bottom: 15px; display: flex; align-items: center; gap: 8px; }
  .loading { display: none; text-align: center; padding: 20px; }
  .loading.show { display: block; }
  .spinner { width: 40px; height: 40px; border: 4px solid #2a2a5a; border-top-color: #4a4aff; border-radius: 50%; animation: spin 0.8s linear infinite; margin: 0 auto 10px; }
  @keyframes spin { to { transform: rotate(360deg); } }
  .stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 15px; margin-bottom: 20px; }
  .stat-card { background: #1a1a3e; border-radius: 10px; padding: 15px; text-align: center; }
  .stat-number { font-size: 28px; font-weight: bold; color: #7b7bff; }
  .stat-label { font-size: 12px; color: #aaa; margin-top: 4px; }
  #autoScanToggle { position: relative; }
</style>
</head>
<body>
<div class="header">
  <h1>🎓 Face ID - Điểm Danh Lớp</h1>
  <p>Hệ thống nhận diện khuôn mặt tự động</p>
</div>
<div class="container">
  <!-- Stats -->
  <div class="stats">
    <div class="stat-card">
      <div class="stat-number" id="totalScanned">0</div>
      <div class="stat-label">Lượt quét</div>
    </div>
    <div class="stat-card">
      <div class="stat-number" id="totalPresent">0</div>
      <div class="stat-label">Có mặt</div>
    </div>
    <div class="stat-card">
      <div class="stat-number" id="totalUnknown">0</div>
      <div class="stat-label">Không nhận ra</div>
    </div>
  </div>

  <!-- Camera -->
  <div class="camera-section">
    <div class="section-title">📷 Camera Nhận Diện</div>
    <div class="camera-wrapper" id="cameraWrapper">
      <video id="video" autoplay muted playsinline></video>
      <canvas id="canvas"></canvas>
      <div class="scan-line" id="scanLine"></div>
    </div>
    <div class="btn-group">
      <button class="btn btn-primary" id="startCamera">📷 Bật Camera</button>
      <button class="btn btn-success" id="captureBtn" disabled>🔍 Nhận Diện</button>
      <button class="btn" id="autoScanToggle" disabled style="background:#2a2a5a">⏱ Tự Động (3s)</button>
      <button class="btn btn-danger" id="stopCamera" disabled>⏹ Tắt Camera</button>
    </div>
  </div>

  <!-- Result -->
  <div class="result-section">
    <div class="section-title">🎯 Kết Quả Nhận Diện</div>
    <div class="loading" id="loading">
      <div class="spinner"></div>
      <p>Đang phân tích khuôn mặt...</p>
    </div>
    <div class="result-card" id="resultCard">
      <div style="font-size:48px" id="resultEmoji">👤</div>
      <div class="result-name" id="resultName">—</div>
      <div class="result-confidence" id="resultConfidence">—</div>
      <div class="confidence-bar"><div class="confidence-fill" id="confidenceFill" style="width:0%"></div></div>
      <div class="top3" id="top3Container"></div>
    </div>
  </div>

  <!-- Attendance Log -->
  <div class="attendance-log">
    <div class="section-title" style="justify-content:space-between">
      <span>📋 Lịch Sử Điểm Danh</span>
      <button class="btn btn-danger" onclick="clearLog()" style="padding:6px 15px;font-size:13px">🗑 Xóa</button>
    </div>
    <table class="log-table">
      <thead>
        <tr>
          <th>#</th><th>Tên</th><th>Độ tin cậy</th><th>Thời gian</th><th>Trạng thái</th>
        </tr>
      </thead>
      <tbody id="logBody"></tbody>
    </table>
  </div>
</div>

<script>
const video = document.getElementById('video');
const canvas = document.getElementById('canvas');
const ctx = canvas.getContext('2d');
let stream = null;
let autoScanInterval = null;
let isAutoScan = false;
let scanCount = 0, presentCount = 0, unknownCount = 0;
const logData = [];

document.getElementById('startCamera').addEventListener('click', async () => {
  try {
    stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480, facingMode: 'user' }, audio: false });
    video.srcObject = stream;
    document.getElementById('captureBtn').disabled = false;
    document.getElementById('autoScanToggle').disabled = false;
    document.getElementById('stopCamera').disabled = false;
    document.getElementById('startCamera').disabled = true;
  } catch (e) {
    alert('Không thể truy cập camera: ' + e.message);
  }
});

document.getElementById('stopCamera').addEventListener('click', stopCamera);
function stopCamera() {
  if (stream) { stream.getTracks().forEach(t => t.stop()); stream = null; }
  if (autoScanInterval) { clearInterval(autoScanInterval); autoScanInterval = null; isAutoScan = false; }
  video.srcObject = null;
  document.getElementById('startCamera').disabled = false;
  document.getElementById('captureBtn').disabled = true;
  document.getElementById('autoScanToggle').disabled = true;
  document.getElementById('stopCamera').disabled = true;
  document.getElementById('autoScanToggle').textContent = '⏱ Tự Động (3s)';
  document.getElementById('autoScanToggle').style.background = '#2a2a5a';
  document.getElementById('cameraWrapper').classList.remove('scanning');
}

document.getElementById('captureBtn').addEventListener('click', captureAndPredict);

document.getElementById('autoScanToggle').addEventListener('click', () => {
  isAutoScan = !isAutoScan;
  const btn = document.getElementById('autoScanToggle');
  const wrapper = document.getElementById('cameraWrapper');
  if (isAutoScan) {
    btn.textContent = '⏹ Dừng Auto';
    btn.style.background = 'linear-gradient(135deg,#fdcb6e,#e17055)';
    wrapper.classList.add('scanning');
    autoScanInterval = setInterval(captureAndPredict, 3000);
  } else {
    btn.textContent = '⏱ Tự Động (3s)';
    btn.style.background = '#2a2a5a';
    wrapper.classList.remove('scanning');
    clearInterval(autoScanInterval);
  }
});

async function captureAndPredict() {
  canvas.width = video.videoWidth || 640;
  canvas.height = video.videoHeight || 480;
  ctx.drawImage(video, 0, 0);
  const imageData = canvas.toDataURL('image/jpeg', 0.8);

  document.getElementById('loading').classList.add('show');
  document.getElementById('resultCard').classList.remove('show');

  try {
    const resp = await fetch('/predict', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ image: imageData })
    });
    const data = await resp.json();
    displayResult(data);
  } catch (e) {
    console.error(e);
  } finally {
    document.getElementById('loading').classList.remove('show');
  }
}

function displayResult(data) {
  const card = document.getElementById('resultCard');
  card.classList.remove('status-known', 'status-unknown');
  card.classList.add('show');

  document.getElementById('resultEmoji').textContent = data.is_known ? '✅' : '❓';
  document.getElementById('resultName').textContent = data.name;
  document.getElementById('resultConfidence').textContent = `Độ tin cậy: ${(data.confidence * 100).toFixed(1)}%`;

  const fill = document.getElementById('confidenceFill');
  fill.style.width = (data.confidence * 100) + '%';
  fill.style.background = data.is_known ? 'linear-gradient(90deg,#00b894,#00cec9)' : 'linear-gradient(90deg,#e17055,#d63031)';
  card.classList.add(data.is_known ? 'status-known' : 'status-unknown');

  // Top 3
  let top3HTML = '<div style="font-size:13px;color:#aaa;margin-bottom:8px">Top dự đoán:</div>';
  data.top3.forEach((p, i) => {
    top3HTML += `<div class="top3-item"><span>${i===0?'🥇':i===1?'🥈':'🥉'} ${p.name}</span><span style="color:#7b7bff">${(p.confidence*100).toFixed(1)}%</span></div>`;
  });
  document.getElementById('top3Container').innerHTML = top3HTML;

  // Update stats & log
  scanCount++;
  if (data.is_known) presentCount++; else unknownCount++;
  document.getElementById('totalScanned').textContent = scanCount;
  document.getElementById('totalPresent').textContent = presentCount;
  document.getElementById('totalUnknown').textContent = unknownCount;
  addLog(data);
}

function addLog(data) {
  const now = new Date().toLocaleTimeString('vi-VN');
  logData.unshift({ name: data.name, confidence: data.confidence, time: now, known: data.is_known });
  const tbody = document.getElementById('logBody');
  const row = document.createElement('tr');
  row.innerHTML = `
    <td>${logData.length}</td>
    <td><strong>${data.name}</strong></td>
    <td>${(data.confidence*100).toFixed(1)}%</td>
    <td>${now}</td>
    <td><span class="badge ${data.is_known?'badge-present':'badge-unknown'}">${data.is_known?'✓ Có mặt':'✗ Không rõ'}</span></td>`;
  tbody.insertBefore(row, tbody.firstChild);
}

function clearLog() {
  document.getElementById('logBody').innerHTML = '';
  logData.length = 0;
  scanCount = presentCount = unknownCount = 0;
  document.getElementById('totalScanned').textContent = '0';
  document.getElementById('totalPresent').textContent = '0';
  document.getElementById('totalUnknown').textContent = '0';
}
</script>
</body>
</html>'''

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        img_array = preprocess_image(data['image'])
        predictions = model.predict(img_array)[0]
        top_idx = int(np.argmax(predictions))
        confidence = float(predictions[top_idx])
        top3 = [{"name": class_labels[int(i)], "confidence": float(predictions[int(i)])}
                for i in np.argsort(predictions)[-3:][::-1]]
        return jsonify({
            "name": class_labels[top_idx] if confidence >= CONFIDENCE_THRESHOLD else "Không nhận ra",
            "confidence": confidence,
            "is_known": confidence >= CONFIDENCE_THRESHOLD,
            "top3": top3
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Chạy server với ngrok
port = 5000
ngrok.set_auth_token("3ERL0z7QnLJGyKCcW4DAaVcdadr_3KvV7MaoMorEMA3zqph3m")
public_url = ngrok.connect(port)
print(f"\n🚀 Web App đang chạy tại: {public_url}")
print("📱 Mở link trên điện thoại hoặc máy tính để dùng!")

app.run(port=port)


🚀 Web App đang chạy tại: NgrokTunnel: "https://hate-angular-possum.ngrok-free.dev" -> "http://localhost:5000"
📱 Mở link trên điện thoại hoặc máy tính để dùng!
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
